Llama parsing on cloude based  

PDF --> upload to Llamacloud --> Llama  processing the PDF --> Parsed (markdown, plain text,  page wide records, Tables, images, execl workbook, Json)

In [3]:
# uv pip install llama_cloud

In [5]:
import os
import io
import re
import json
from pathlib import Path
from typing import Any

In [6]:
import pandas as pd
import requests
from dotenv import load_dotenv
import llama_cloud

In [2]:
from llama_cloud import LlamaCloud

In [ ]:
# ============================================================
# 1. File Paths
# ============================================================

BASE_DIR = Path.cwd()
Complex_RAG = BASE_DIR / "2_Complex_RAG.pdf"
print(Complex_RAG)

PDF_PATH = Path(Complex_RAG)

OUTPUT_DIR = Path(BASE_DIR / "2_Llama_Parsed_Output")

IMAGE_DIR = OUTPUT_DIR / "extracted_images"
SCREENSHOT_DIR = IMAGE_DIR / "screenshots"
EMBEDDED_IMAGE_DIR = IMAGE_DIR / "embedded"
LAYOUT_IMAGE_DIR = IMAGE_DIR / "layout"
OTHER_IMAGE_DIR = IMAGE_DIR / "other"
TABLE_DIR = OUTPUT_DIR / "extracted_tables"


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
LAYOUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OTHER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print("PDF found:", PDF_PATH)
print("Output directory:", OUTPUT_DIR)


SyntaxError: invalid syntax (3238062077.py, line 7)

In [10]:
load_dotenv()

LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")

if not LLAMA_CLOUD_API_KEY:
    raise ValueError(
        "LLAMA_CLOUD_API_KEY was not found.\n"
        "Create a .env file and add:\n"
        "LLAMA_CLOUD_API_KEY=llx-your-api-key"
    )

client = LlamaCloud(
    api_key=LLAMA_CLOUD_API_KEY
)

print("LlamaCloud client initialized successfully.")

LlamaCloud client initialized successfully.


In [11]:
# ============================================================
# 3. Helper Functions
# ============================================================

def safe_filename(filename: str) -> str:
    """
    Removes unsafe characters from a filename.
    """
    filename = Path(filename).name
    return re.sub(r'[<>:"/\\|?*]', "_", filename)


def object_to_dict(obj: Any) -> Any:
    """
    Safely converts an SDK/Pydantic object into a dictionary.
    """
    if obj is None:
        return None

    if isinstance(obj, dict):
        return {
            key: object_to_dict(value)
            for key, value in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [object_to_dict(value) for value in obj]

    if hasattr(obj, "to_dict"):
        return object_to_dict(obj.to_dict())

    if hasattr(obj, "model_dump"):
        return object_to_dict(obj.model_dump(mode="json"))

    if hasattr(obj, "__dict__"):
        return {
            key: object_to_dict(value)
            for key, value in vars(obj).items()
            if not key.startswith("_")
        }

    return obj


def download_file(
    url: str,
    output_path: Path,
    timeout: int = 120
) -> bool:
    """
    Downloads a file from a presigned URL.
    """
    try:
        response = requests.get(
            url,
            timeout=timeout,
            stream=True
        )

        response.raise_for_status()

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with output_path.open("wb") as file:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

        return True

    except Exception as error:
        print(
            f"Download failed for {output_path.name}: "
            f"{type(error).__name__}: {error}"
        )
        return False


def get_image_output_directory(category: str | None) -> Path:
    """
    Selects the correct folder according to image category.
    """
    category = (category or "other").lower()

    if category == "screenshot":
        return SCREENSHOT_DIR

    if category == "embedded":
        return EMBEDDED_IMAGE_DIR

    if category == "layout":
        return LAYOUT_IMAGE_DIR

    return OTHER_IMAGE_DIR

In [12]:
print("\nUploading PDF to LlamaCloud...")

uploaded_file = client.files.create(
    file=PDF_PATH,
    purpose="parse"
)

print("Upload completed.")
print("Uploaded file ID:", uploaded_file.id)


Uploading PDF to LlamaCloud...
Upload completed.
Uploaded file ID: 5574a3fd-3b66-4b1f-9742-d83ea62f6744


In [13]:
# ============================================================
# 5. Parse PDF
# ============================================================

print("\nParsing PDF with LlamaParse...")

result = client.parsing.parse(
    file_id=uploaded_file.id,

    # Suitable for complex documents containing
    # tables, images, scans and mixed layouts.
    tier="agentic",

    version="latest",

    output_options={
        # Preserve tables in Markdown output.
        "markdown": {
            "tables": {
                "output_tables_as_markdown": True
            }
        },

        # Generate an Excel workbook containing tables.
        "tables_as_spreadsheet": {
            "enable": True
        },

        # Save complete page screenshots,
        # original embedded images and detected layout regions.
        "images_to_save": [
            "screenshot",
            "embedded",
            "layout"
        ],
    },

    processing_options={
        # OCR language configuration.
        "ocr_parameters": {
            "languages": ["en"]
        }
    },

    # Controls which results are returned.
    expand=[
        "text",
        "text_full",
        "markdown",
        "markdown_full",
        "items",
        "metadata",
        "job_metadata",
        "images_content_metadata",
        "xlsx_content_metadata",
    ],

    verbose=True,
    timeout=600,
)

print("\nParsing completed.")
print("Job ID:", result.job.id)
print("Job status:", result.job.status)


Parsing PDF with LlamaParse...

Parsing completed.
Job ID: pjb-0l6ddp4ngho2ww2d7xffqsd9sql3
Job status: COMPLETED


In [14]:
raw_result = object_to_dict(result)

raw_result_path = OUTPUT_DIR / "llamaparse_raw_result.json"

with raw_result_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        raw_result,
        file,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("Raw JSON saved:", raw_result_path)

Raw JSON saved: D:\MAHA\AIPro\AgenticAI\myAgenticAI_7am_May26\Datallamaparsed_output\llamaparse_raw_result.json
